<a href="https://colab.research.google.com/github/sachinarkesh/Data-Visualization/blob/main/week03_lecture03_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import glob, os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
paths = ['../data/co2_emissions.csv'] + glob.glob('/content/co2_emissions.csv', recursive=True)
path = next(p for p in paths if os.path.exists(p))
print("Using:", path)
df = pd.read_csv(path)
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())

Using: /content/co2_emissions.csv
Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [ ]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))

Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


In [ ]:
# Task 1 — CO2 per capita, all countries, one highlighted
HIGHLIGHT = 'China'          # <- your choice
HIGHLIGHT_COLOR = '#D62728'
GREY = '#EEEEEE'
METRIC = 'CO2_per_capita'

d = df.sort_values(['Country', 'Year'])
hl = d[d['Country'] == HIGHLIGHT]
start, end = hl.iloc[0], hl.iloc[-1]
pct = (end[METRIC] / start[METRIC] - 1) * 100
verb = 'rose' if pct >= 0 else 'fell'

fig = go.Figure()

# Context: every other country in thin grey
for country, g in d[d['Country'] != HIGHLIGHT].groupby('Country'):
    fig.add_trace(go.Scatter(x=g['Year'], y=g[METRIC], mode='lines',
                             line=dict(color=GREY, width=1),
                             hoverinfo='skip', showlegend=False))

# Highlighted country on top
fig.add_trace(go.Scatter(x=hl['Year'], y=hl[METRIC], mode='lines',
                         line=dict(color=HIGHLIGHT_COLOR, width=4),
                         showlegend=False,
                         hovertemplate='%{x}: %{y:.2f} t per person<extra></extra>'))

# Direct label at the end of the line (no legend)
fig.add_annotation(x=end['Year'], y=end[METRIC],
                   text=f"<b>{HIGHLIGHT}</b><br>{end[METRIC]:.2f} t",
                   showarrow=False, xanchor='left', xshift=8,
                   font=dict(color=HIGHLIGHT_COLOR, size=14))

fig.update_layout(
    title=dict(text=f"<b>{HIGHLIGHT}'s CO2 per capita {verb} {abs(pct):.0f}% "
                    f"between {int(start['Year'])} and {int(end['Year'])}</b>",
               x=0, xanchor='left', font=dict(size=20)),
    template='simple_white', plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(l=60, r=130, t=90, b=60), width=900, height=520,
    xaxis=dict(title=None, showgrid=False, showline=False, tickangle=0, dtick=2, ticks=''),
    yaxis=dict(title='CO2 per capita (tonnes per person)', showgrid=True,
               gridcolor='#F2F2F2', gridwidth=0.5, showline=False, zeroline=False, ticks=''),
    showlegend=False
)
fig.show()

In [ ]:
# Task 2 — CO2 emissions, Asian countries, one highlighted
HIGHLIGHT = 'China'
HIGHLIGHT_COLOR = '#D62728'
GREY = '#EEEEEE'

asia = df[df['Region'] == 'Asia'].sort_values(['Country', 'Year'])
hl = asia[asia['Country'] == HIGHLIGHT]
start, end = hl.iloc[0], hl.iloc[-1]
multiple = end['CO2_Mt'] / start['CO2_Mt']

fig = go.Figure()

for country, g in asia[asia['Country'] != HIGHLIGHT].groupby('Country'):
    fig.add_trace(go.Scatter(x=g['Year'], y=g['CO2_Mt'], mode='lines',
                             line=dict(color=GREY, width=1.5),
                             hoverinfo='skip', showlegend=False))

fig.add_trace(go.Scatter(x=hl['Year'], y=hl['CO2_Mt'], mode='lines',
                         line=dict(color=HIGHLIGHT_COLOR, width=4),
                         showlegend=False,
                         hovertemplate='%{x}: %{y:,.0f} Mt<extra></extra>'))

fig.add_annotation(x=end['Year'], y=end['CO2_Mt'],
                   text=f"<b>{HIGHLIGHT}</b><br>{end['CO2_Mt']:,.0f} Mt",
                   showarrow=False, xanchor='left', xshift=8,
                   font=dict(color=HIGHLIGHT_COLOR, size=14))

fig.update_layout(
    title=dict(text=f"<b>{HIGHLIGHT}'s CO2 emissions grew {multiple:.1f}x since "
                    f"{int(start['Year'])}, pulling far ahead of the rest of Asia</b>",
               x=0, xanchor='left', font=dict(size=20)),
    template='simple_white', plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(l=60, r=130, t=90, b=60), width=900, height=520,
    xaxis=dict(title=None, showgrid=False, showline=False, tickangle=0, dtick=2, ticks=''),
    yaxis=dict(title='CO2 emissions (Mt)', showgrid=True, gridcolor='#F2F2F2',
               gridwidth=0.5, showline=False, zeroline=False, ticks='', tickformat=','),
    showlegend=False
)
fig.show()